# Kanana 태깅·추출 LoRA / QLoRA 학습 (Colab)

**목적:** 같은 데이터·같은 설정으로 LoRA와 QLoRA를 각각 학습해서, **품질 대비 비용**이 더 좋은 쪽을 고른다.

| 순서 | 셀 | 하는 일 |
|---|---|---|
| 1 | GPU 확인 | 어떤 GPU가 붙었는지 확인 (T4 / L4 / A100) |
| 2 | 설치 | 학습 라이브러리 설치 |
| 3 | **⚙️ 설정** | **여기만 바꾼다.** `MODE = "lora"` 또는 `"qlora"` |
| 4 | 드라이브 연결 | 데이터 읽기, 결과 저장 위치 |
| 5 | 데이터 확인 | 형식 검사 + 토큰 길이 확인 |
| 6 | 모델 로드 | MODE에 따라 16비트 또는 4비트로 올림 |
| 7 | 학습 | 학습 + **메모리·시간·비용 자동 기록** |
| 8 | 간단 점검 | 검증 데이터 몇 건으로 JSON 형식이 잘 나오는지 확인 |
| 9 | 비교표 | 지금까지 돌린 실행을 표로 비교 |
| 10 | 서빙 안내 | vLLM에 어댑터를 붙여 Langfuse 평가로 넘어가는 방법 |

**두 모드 비교 방법:** 3번 셀에서 `MODE = "lora"`로 3→9 실행 → **런타임 다시 시작** → `MODE = "qlora"`로 다시 3→9 실행. 다른 값은 건드리지 않는다.

> 런타임을 다시 시작하는 이유: 같은 세션에서 두 번째 모델을 올리면 앞 모델의 GPU 메모리가 남아 최대 메모리 기록이 틀어진다.

**학습 데이터 형식** (`{TASK}_train.jsonl`, `{TASK}_valid.jsonl`, 한 줄에 한 건):
```json
{"messages": [
  {"role": "system", "content": "<운영 TAG_PROMPT 또는 추출 프롬프트 그대로>"},
  {"role": "user", "content": "질문: ...\n답변: ..."},
  {"role": "assistant", "content": "{\"primary\": [\"anxiety\"], \"secondary\": [], \"off_topic\": false}"}
]}
```
- system·user는 **운영 앱이 보내는 것과 글자 하나까지 같아야** 한다. 다르면 학습한 어댑터가 운영에서 제 성능을 못 낸다.
- assistant는 teacher 라벨(JSON 문자열). **손실은 assistant 부분에만** 걸린다.
- ⚠️ Langfuse 평가셋(regression·calibration·blind_holdout)의 문항은 학습에 넣지 않는다.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. 설치
Kanana-2 설정 파일이 transformers 4.57로 만들어져 있어서 4.x 최신으로 맞춘다. 설치 후 경고가 떠도 다음 셀로 넘어가면 된다.

In [ ]:
%pip install -q "transformers>=4.57,<5" "peft>=0.17" "trl>=0.24" "bitsandbytes>=0.47" "accelerate>=1.10" "datasets>=4.0"

## 3. ⚙️ 설정 — 이 셀만 바꾼다

| 값 | 뜻 |
|---|---|
| `MODE` | `"lora"`: 원본 16비트 / `"qlora"`: 원본 4비트(NF4) |
| `TASK` | `"tagging"` 또는 `"extraction"` |
| `GPU_HOURLY_USD` | 시간당 GPU 단가(달러). 비용 칸 계산용. 모르면 0으로 두고 나중에 표에서 계산 |
| `HP` | 학습 설정. **LoRA·QLoRA 비교 중에는 바꾸지 않는다** |

In [ ]:
MODE = "qlora"  # "lora" 또는 "qlora"  ← 비교할 때 이것만 바꾼다
TASK = "tagging"  # "tagging" 또는 "extraction"

MODEL_ID = "kakaocorp/kanana-2-3b-instruct"
DATA_DIR = "/content/drive/MyDrive/ktb-ai/train_data"  # {TASK}_train.jsonl, {TASK}_valid.jsonl 위치
OUT_ROOT = "/content/drive/MyDrive/ktb-ai/adapters"  # 어댑터와 기록이 쌓이는 곳
GPU_HOURLY_USD = 0.0  # 예: Colab Pro 환산 단가, 클라우드 T4 약 0.35, L4 약 0.7
SEED = 42

# 작업별 학습 설정. max_len 은 pilot200 최대 길이(태깅 975, 추출 3,257 토큰)에 여유를 둔 값.
# 추출은 대화 전체가 들어가서 길고 배치를 줄인다.
HP = {
    "tagging": dict(r=16, alpha=32, dropout=0.05, epochs=3, lr=2e-4, batch=8, grad_accum=2, max_len=1280),
    "extraction": dict(r=16, alpha=32, dropout=0.05, epochs=3, lr=2e-4, batch=1, grad_accum=16, max_len=4096),
}[TASK]
# 생성 길이: 운영 앱의 max_tokens 와 맞춘다 (태깅 120)
MAX_NEW_TOKENS = {"tagging": 120, "extraction": 1024}[TASK]
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

assert MODE in ("lora", "qlora"), MODE
assert TASK in ("tagging", "extraction"), TASK
print(f"MODE={MODE} TASK={TASK} HP={HP}")

## 4. 드라이브 연결

In [ ]:
import os

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Colab이 아니므로 드라이브 연결을 건너뜀")

os.makedirs(OUT_ROOT, exist_ok=True)
TRAIN_FILE = f"{DATA_DIR}/{TASK}_train.jsonl"
VALID_FILE = f"{DATA_DIR}/{TASK}_valid.jsonl"
for path in (TRAIN_FILE, VALID_FILE):
    print(("✅ " if os.path.exists(path) else "❌ 없음: ") + path)

## 5. 데이터 확인
형식이 틀린 줄, assistant가 JSON이 아닌 줄, `max_len`보다 긴 줄을 미리 잡는다. 긴 줄은 학습 때 잘려서 정답 JSON이 사라지므로 **0건이어야** 한다.

In [ ]:
import json

from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
ds = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VALID_FILE})


def _ids(messages):
    out = tok.apply_chat_template(messages, tokenize=True)
    return out["input_ids"] if hasattr(out, "keys") else out


problems = []
lengths = []
for split in ("train", "validation"):
    for i, row in enumerate(ds[split]):
        roles = [m["role"] for m in row["messages"]]
        if roles != ["system", "user", "assistant"]:
            problems.append(f"{split}[{i}] 역할 순서 {roles}")
            continue
        try:
            json.loads(row["messages"][-1]["content"])
        except json.JSONDecodeError:
            problems.append(f"{split}[{i}] assistant가 JSON이 아님")
        n = len(_ids(row["messages"]))
        lengths.append(n)
        if n > HP["max_len"]:
            problems.append(f"{split}[{i}] 토큰 {n} > max_len {HP['max_len']}")

lengths.sort()
print(f"train {len(ds['train'])}건 / validation {len(ds['validation'])}건")
print(f"토큰 길이: 최대 {lengths[-1]}, p95 {lengths[int(len(lengths) * 0.95) - 1]}, 중앙값 {lengths[len(lengths) // 2]}")
print(f"문제 {len(problems)}건")
for p in problems[:20]:
    print(" -", p)
assert not problems, "위 문제를 고친 뒤 다시 실행하세요"

## 6. 모델 로드
- `lora`: 원본을 16비트로 올린다 (Kanana 약 8GB).
- `qlora`: 원본을 4비트 NF4로 올린다 (약 3GB). 계산은 16비트로 한다.
- T4는 BF16을 지원하지 않아 FP16을 쓴다. L4·A100이면 자동으로 BF16을 쓴다.

In [ ]:
import time

import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

HAS_CUDA = torch.cuda.is_available()
USE_BF16 = HAS_CUDA and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16 if HAS_CUDA else torch.float32
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else "cpu"


def gib(n_bytes):
    return round(n_bytes / 2**30, 2)


if HAS_CUDA:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
load_start = time.perf_counter()

if MODE == "qlora":
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE,
    )
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant, dtype=DTYPE, device_map={"": 0})
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False}
    )
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map={"": 0} if HAS_CUDA else None)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()

model.config.use_cache = False
model = get_peft_model(
    model,
    LoraConfig(
        r=HP["r"],
        lora_alpha=HP["alpha"],
        lora_dropout=HP["dropout"],
        target_modules=TARGET_MODULES,
        bias="none",
        task_type="CAUSAL_LM",
    ),
)
# FP16 혼합정밀도는 학습하는 가중치가 FP32 여야 한다 (아니면 "Attempting to unscale FP16 gradients" 오류)
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

LOAD_SECONDS = round(time.perf_counter() - load_start, 1)
LOAD_PEAK_GIB = gib(torch.cuda.max_memory_allocated()) if HAS_CUDA else 0.0
print(f"{GPU_NAME} · dtype={DTYPE} · 로드 {LOAD_SECONDS}s · 로드 후 메모리 {LOAD_PEAK_GIB} GiB")
model.print_trainable_parameters()

## 7. 학습 + 자동 기록
끝나면 `OUT_ROOT/<실행 이름>/` 에 어댑터와 `run_metrics.json` 이 저장되고, `OUT_ROOT/runs.jsonl` 에 한 줄이 추가된다.

| 기록 항목 | 뜻 |
|---|---|
| `train_peak_gib` | 학습 중 최대 GPU 메모리 (실제로 쓴 양) |
| `train_reserved_gib` | PyTorch가 잡아 둔 메모리 (`nvidia-smi`에 보이는 값에 가까움) |
| `train_seconds` | 학습 시간 |
| `train_cost_usd` | 학습 시간 × `GPU_HOURLY_USD` |
| `eval_loss` | 검증 손실 (가장 좋은 epoch 기준) |

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from trl import SFTConfig, SFTTrainer

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M")
RUN_NAME = f"kanana-{TASK}-{MODE}-r{HP['r']}-s{SEED}-{stamp}"
RUN_DIR = Path(OUT_ROOT) / RUN_NAME

config = SFTConfig(
    output_dir=str(RUN_DIR / "checkpoints"),
    num_train_epochs=HP["epochs"],
    per_device_train_batch_size=HP["batch"],
    per_device_eval_batch_size=HP["batch"],
    gradient_accumulation_steps=HP["grad_accum"],
    learning_rate=HP["lr"],
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    optim="adamw_torch",  # 두 모드 모두 같은 옵티마이저 (차이는 원본 정밀도 하나만)
    max_length=HP["max_len"],
    assistant_only_loss=True,  # 정답 JSON 부분에만 손실
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=HAS_CUDA and not USE_BF16,
    bf16=USE_BF16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=5,
    seed=SEED,
    report_to="none",
)
trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    processing_class=tok,
)

if HAS_CUDA:
    torch.cuda.reset_peak_memory_stats()
train_start = time.perf_counter()
train_result = trainer.train()
TRAIN_SECONDS = round(time.perf_counter() - train_start, 1)

eval_metrics = trainer.evaluate()
adapter_dir = RUN_DIR / "adapter"
trainer.save_model(str(adapter_dir))
tok.save_pretrained(str(adapter_dir))
adapter_mb = round(sum(f.stat().st_size for f in adapter_dir.glob("*.safetensors")) / 2**20, 1)

RUN_METRICS = {
    "run_name": RUN_NAME,
    "mode": MODE,
    "task": TASK,
    "model_id": MODEL_ID,
    "gpu": GPU_NAME,
    "dtype": str(DTYPE).replace("torch.", ""),
    "seed": SEED,
    "hp": HP,
    "train_rows": len(ds["train"]),
    "valid_rows": len(ds["validation"]),
    "load_seconds": LOAD_SECONDS,
    "load_peak_gib": LOAD_PEAK_GIB,
    "train_seconds": TRAIN_SECONDS,
    "train_peak_gib": gib(torch.cuda.max_memory_allocated()) if HAS_CUDA else 0.0,
    "train_reserved_gib": gib(torch.cuda.max_memory_reserved()) if HAS_CUDA else 0.0,
    "train_samples_per_second": train_result.metrics.get("train_samples_per_second"),
    "train_loss": round(train_result.training_loss, 4),
    "eval_loss": round(eval_metrics["eval_loss"], 4),
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "train_cost_usd": round(TRAIN_SECONDS / 3600 * GPU_HOURLY_USD, 4),
    "adapter_mb": adapter_mb,
    "adapter_dir": str(adapter_dir),
    "finished_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
(RUN_DIR / "run_metrics.json").write_text(json.dumps(RUN_METRICS, ensure_ascii=False, indent=2))
with open(Path(OUT_ROOT) / "runs.jsonl", "a") as f:
    f.write(json.dumps(RUN_METRICS, ensure_ascii=False) + "\n")
print(json.dumps(RUN_METRICS, ensure_ascii=False, indent=2))

## 8. 간단 점검 (정식 평가 아님)
검증 데이터 몇 건으로 **JSON이 제대로 나오는지**만 본다. 정식 품질 점수(태깅 F1, 추출 차원 판정 정확도)는 10번처럼 vLLM에 올려 Langfuse 평가 스크립트로 잰다.

여기 나오는 생성 속도는 HF `generate` 기준이라 **서빙 속도가 아니다.** 비교에 쓰지 않는다.

In [ ]:
import re

SAMPLE_N = 10
THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)

model.eval()
model.config.use_cache = True
parsed = exact = 0
rows = ds["validation"].select(range(min(SAMPLE_N, len(ds["validation"]))))
for row in rows:
    prompt = tok.apply_chat_template(row["messages"][:-1], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    prompt = {k: v.to(model.device) for k, v in prompt.items()}
    with torch.no_grad():
        out = model.generate(**prompt, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
    text = THINK_RE.sub("", tok.decode(out[0][prompt["input_ids"].shape[1] :], skip_special_tokens=True)).strip()
    try:
        got = json.loads(text)
        parsed += 1
        exact += got == json.loads(row["messages"][-1]["content"])
    except json.JSONDecodeError:
        print("JSON 아님:", text[:200])
print(f"JSON 파싱 {parsed}/{len(rows)} · 정답과 완전히 같음 {exact}/{len(rows)}")

## 9. 비교표
`runs.jsonl` 에 쌓인 실행을 모아 본다. LoRA·QLoRA를 둘 다 돌린 뒤 실행한다.
품질 점수 칸(태깅 F1, 추출 정확도)은 Langfuse 평가 결과를 보고 보고서 표에 채운다.

In [ ]:
import pandas as pd

runs = pd.read_json(Path(OUT_ROOT) / "runs.jsonl", lines=True)
columns = [
    "run_name", "mode", "task", "gpu", "dtype", "train_peak_gib", "train_reserved_gib",
    "train_seconds", "train_cost_usd", "eval_loss", "adapter_mb",
]
display(runs[runs["task"] == TASK][columns].sort_values("run_name"))

## 10. vLLM에 어댑터를 붙여 Langfuse 평가하기
학습 런타임을 **다시 시작**한 뒤(메모리 비우기) 기준선 측정 노트북의 서버 실행 셀에 아래 옵션만 더한다.

```bash
vllm serve kakaocorp/kanana-2-3b-instruct --dtype half \
  --enable-lora --max-lora-rank 16 \
  --lora-modules kanana-tagging-lora=<LoRA adapter 경로> kanana-tagging-qlora=<QLoRA adapter 경로>
```

- 두 어댑터를 **한 서버에 동시에** 올릴 수 있다. 원본 정밀도가 같으니 서빙 조건이 똑같아 공정하다.
- 맥에서 평가할 때 `LLM_MODEL` 을 어댑터 이름으로 바꿔 실행한다.
  ```bash
  LLM_MODEL=kanana-tagging-qlora uv run python scripts/run_langfuse_calibration.py \
    --dataset quality/persona/onboarding-tagging --split regression \
    --run-name kanana-tagging-qlora-T4fp16-regression-<날짜>
  ```
- 앱은 `LLM_MODEL` 하나로 모든 기능을 부르므로, **`--dataset` 을 어댑터 작업과 맞춰** 한 번에 하나씩 돌린다 (태깅 어댑터 → onboarding-tagging, 추출 어댑터 → build).
- 4비트 원본(AWQ 등)으로 서빙할 계획이면 그 원본에서도 한 번 더 평가한다. 16비트에서 학습한 LoRA 어댑터는 4비트 원본에 붙이면 품질이 떨어질 수 있다.